# ITO5201 – Assessment 1: Section 4
## Logistic Regression versus Bayes Classifier
**Student:** Johannes Coetzee  
**Student Number:** 36384852

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import zero_one_loss, log_loss
from sklearn.model_selection import train_test_split

*Source: `BayesianClassifier` adapted from Activity 3.3 (`Modules/3/Activities/Activity.3.3_solutions.ipynb`). `LogisticRegression` is the discriminative baseline whose comparison with the Bayesian generative classifier is set up as the Assignment 1 goal in Activity 3.2 (`Modules/3/Activities/Activity.3.2.ipynb`): "In Assignment 1, you will compare logistic regression with the Bayesian generative classifier."*

`numpy` and `matplotlib` are the standard numerical and plotting libraries. `multivariate_normal` from `scipy.stats` provides the multivariate Gaussian PDF needed inside `BayesianClassifier.predict_proba` to evaluate $p(\mathbf{x} \mid C = c)$. `load_breast_cancer` supplies the Wisconsin breast cancer dataset: 569 samples, 30 real-valued features, binary class label (malignant/benign). `LogisticRegression` is the sklearn discriminative baseline. `zero_one_loss` and `log_loss` from `sklearn.metrics` are used throughout Activities 3.2 and 3.3 as the standard classification error metrics — `zero_one_loss` computes the fraction of misclassified samples (error rate), `log_loss` evaluates the probabilistic quality of `predict_proba` outputs. `train_test_split` from `sklearn.model_selection` performs the 80/20 dataset split.

---
## Question 7 – Discriminative vs Generative Models
### Bayes Classifier (from Activity 3.3)

Copy the Bayes classifier implementation from Activity 3.3 below. Include all three variants:
- Naive Bayes (no shared covariance)
- Full covariance, shared
- Full covariance, not shared

*Source: Activity 3.3 (`Modules/3/Activities/Activity.3.3_solutions.ipynb`), Tasks A–D.*

**`__init__`** stores two flags that together select one of three model variants:
* `cond_ind=True, shared_cov=False` → **Naive Bayes**: diagonal per-class covariances (features conditionally independent given class).
* `cond_ind=False, shared_cov=True` → **BC Shared Covariance** (equivalent to Linear Discriminant Analysis): one full covariance matrix shared across all classes.
* `cond_ind=False, shared_cov=False` → **BC Non-Shared Covariance** (equivalent to Quadratic Discriminant Analysis): a separate full covariance matrix per class.

**`fit`** computes class statistics:
* `np.unique(y, return_counts=True)` — returns sorted class labels and counts in one call. Preferred over a loop because the output arrays integrate directly with numpy indexing throughout the method.
* `self.class_priors_ = class_counts / len(y)` — empirical prior $\hat{\pi}_k = N_k / N$.
* `x[mask].var(axis=0)` when `cond_ind=True` — per-feature *variance* (diagonal of the covariance matrix). Using `std` would require squaring afterward.
* `np.cov(x[mask].T, bias=True)` when `cond_ind=False` — `np.cov` expects `(variables, observations)`, so `.T` transposes from `(N_k, p)` to `(p, N_k)`. `bias=True` divides by $N_k$ (MLE) rather than $N_k - 1$ (Bessel's corrected sample covariance), consistent with the mathematical derivation (see Activity 3.3, Task C).
* `np.moveaxis(self.cond_covs_, 0, -1).dot(self.class_priors_)` when `shared_cov=True` — computes $\hat{\Sigma} = \sum_k \hat{\pi}_k \hat{\Sigma}_k$. Moving axis 0 to -1 turns `(k, p, p)` into `(p, p, k)`, and `.dot(class_priors_)` contracts along the last axis to give the `(p, p)` weighted-average covariance (Activity 3.3, Task D).

**`predict_proba`** applies Bayes' theorem:
* `multivariate_normal.pdf(..., allow_singular=True)` — `allow_singular=True` is essential when $N_k \leq p$: the class covariance has rank at most $N_k - 1 < p$ and is singular. Without this flag, scipy raises a `LinAlgError`. Activity 3.3 explicitly includes this flag for robustness.
* `cond_probs @ self.class_priors_` — marginal $p(\mathbf{x}) = \sum_k p(\mathbf{x} \mid C=k)\hat{\pi}_k$ via the law of total probability; `(m, k) @ (k,)` → `(m,)`.
* `np.divide(..., where=marginal_probs > 0, out=np.zeros(...))` — skips samples where the marginal is zero using numpy's `where` argument. A plain `/` would produce `nan`/`inf` for those samples.

**`predict`** returns `self.classes_[np.argmax(...)]` — indexing into `classes_` ensures the returned labels match the original label space (which may not be `{0, 1, …}`).

**`decision_function`** — added from Activity 3.3. Returns the log-odds $\log p(c \mid \mathbf{x}) / p(\neg c \mid \mathbf{x})$ per class, providing a real-valued discriminant score. The `np.maximum(..., 1e-300)` guards against `log(0)`. Note: Activity 3.3's multi-class branch contained a typo — `np.zeros(len(x), self.k_)` instead of `np.zeros((len(x), self.k_))` (shape must be a tuple); this is corrected here.

**`generate`** — from Activity 3.3. Samples new input vectors from the fitted class-conditional Gaussian $p(\mathbf{x} \mid C = c)$ using `multivariate_normal.rvs`. This demonstrates the generative nature of the model: unlike logistic regression, the Bayesian classifier explicitly models $p(\mathbf{x} \mid c)$ and can therefore synthesise new samples per class.

In [ ]:
# Source: Adapted from Activity 3.3 (Modules/3/Activities/Activity.3.3_solutions.ipynb)
class BayesianClassifier:
    """
    Bayesian classifier with multivariate normal class conditionals.
    Variants controlled by:
      shared_cov : if True, use a single weighted-average covariance for all classes
      cond_ind   : if True, use diagonal covariance (Naive Bayes assumption)
    """

    def __init__(self, shared_cov=True, cond_ind=True):
        self.shared_cov = shared_cov
        self.cond_ind   = cond_ind

    def fit(self, x, y):
        self.classes_, class_counts = np.unique(y, return_counts=True)
        self.n_, self.p_ = x.shape
        self.k_ = len(self.classes_)
        self.cond_means_ = np.zeros((self.k_, self.p_))
        self.cond_covs_  = np.zeros((self.k_, self.p_, self.p_))

        self.class_priors_ = class_counts / len(y)
        for c in range(self.k_):
            mask = y == self.classes_[c]
            self.cond_means_[c] = x[mask].mean(axis=0)
            if self.cond_ind:
                np.fill_diagonal(self.cond_covs_[c], x[mask].var(axis=0))
            else:
                self.cond_covs_[c] = np.cov(x[mask].T, bias=True)

        if self.shared_cov:
            shared = np.moveaxis(self.cond_covs_, 0, -1).dot(self.class_priors_)
            self.cond_covs_[:] = shared

        return self

    def predict_proba(self, x):
        m = len(x)
        cond_probs = np.zeros((m, self.k_))
        for c in range(self.k_):
            cond_probs[:, c] = multivariate_normal.pdf(
                x, self.cond_means_[c], self.cond_covs_[c], allow_singular=True
            )
        marginal_probs = cond_probs @ self.class_priors_
        probs = np.divide(
            (cond_probs * self.class_priors_).T,
            marginal_probs,
            where=marginal_probs > 0,
            out=np.zeros((self.k_, m))
        ).T
        return probs

    def predict(self, x):
        return self.classes_[np.argmax(self.predict_proba(x), axis=1)]

    def decision_function(self, x):
        # Source: Activity 3.3 — returns log-odds (log p(c|x) / p(not c|x)) per class
        probs = self.predict_proba(x)
        if self.k_ == 2:
            # Binary case: scalar log-odds for the positive class
            return np.log(probs[:, 1] / np.maximum(probs[:, 0], 1e-300))
        else:
            # Multi-class: one log-odds value per class
            # Note: Activity 3.3 had a typo — np.zeros(len(x), self.k_) is wrong;
            # np.zeros((len(x), self.k_)) is correct (shape must be a tuple).
            res = np.zeros((len(x), self.k_))
            for c in range(self.k_):
                res[:, c] = np.log(probs[:, c] / np.maximum(1 - probs[:, c], 1e-300))
            return res

    def generate(self, n, c, random_state=None):
        # Source: Activity 3.3 — sample new inputs from the fitted class-conditional distribution
        return multivariate_normal.rvs(
            self.cond_means_[c], self.cond_covs_[c], size=n, random_state=random_state
        )

*Source: Activity 3.2 (`Modules/3/Activities/Activity.3.2.ipynb`) sets up this exact comparison: "In Assignment 1, you will compare logistic regression with the Bayesian generative classifier." Activity 3.3 (`Modules/3/Activities/Activity.3.3_solutions.ipynb`), Task E, provides the multi-model evaluation pattern via `plot_model_performances`.*

`load_breast_cancer()` returns a Bunch object; `.data` is the $(569 \times 30)$ feature matrix and `.target` is the binary label vector. `train_test_split(..., train_size=0.8, shuffle=True, random_state=42)` produces an 80/20 split with a fixed seed.

The `models_q7` dict maps human-readable names to fitted model objects. Iterating over the dict ensures all four models see exactly the same training and test data. For each model:
* `model.fit(X_train, y_train)` — trains on the 80% training split.
* `model.predict(X_train)` and `model.predict(X_test)` — generate class-label arrays.
* `np.mean(... == y_train)` — computes accuracy as the fraction of correct predictions. (The equivalent in Activity 3.3 uses `zero_one_loss` for the error rate = `1 - accuracy`; here accuracy is reported for readability.)

### Q7.I – Load Breast Cancer Dataset and Fit All Models
*GitHub issue: #29*

`training_sizes = list(range(5, 505, 5))` defines 100 candidate training sizes from 5 to 500 in steps of 5. `valid_sizes` filters out any size that exceeds `len(X_train)` — with an 80/20 split the training pool is ~455 samples, so $N=500$ is excluded.

**`model_factories`** stores **lambda functions** (zero-argument callables) rather than model instances. This ensures each repetition gets a freshly initialised, unfitted model — reusing the same instance would accumulate state across repetitions.

The double loop:
* Outer loop over `valid_sizes` — for each training size $N$, all four models are evaluated across `n_reps=10` independent repetitions.
* Inner repetition loop — `train_test_split(X_train, y_train, train_size=N, shuffle=True)` draws a fresh random subset of size $N$ from the full training pool (no fixed seed, so each repetition is independent). All four models share the same $N$ training points within a repetition.
* `factory()` creates a fresh model instance; `.fit(X_tr, y_tr)` trains it; error rates are `1 - accuracy`.
* After `n_reps` repetitions, the mean error rates across repetitions are appended to `results[name]['train']` and `results[name]['test']`, one value per training size.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, shuffle=True, random_state=42
)

models_q7 = {
    'Logistic Regression':  LogisticRegression(max_iter=10_000),
    'Naive Bayes':          BayesianClassifier(shared_cov=False, cond_ind=True),
    'BC Shared Cov':        BayesianClassifier(shared_cov=True,  cond_ind=False),
    'BC Non-Shared Cov':    BayesianClassifier(shared_cov=False, cond_ind=False),
}

print(f"{'Model':<25}  {'Train acc':>10}  {'Test acc':>10}")
print("-" * 50)
for name, model in models_q7.items():
    model.fit(X_train, y_train)
    train_acc = np.mean(model.predict(X_train) == y_train)
    test_acc  = np.mean(model.predict(X_test)  == y_test)
    print(f"{name:<25}  {train_acc:>10.4f}  {test_acc:>10.4f}")

A single shared figure with one axis plots all eight curves (four models × two splits). The design choices:
* `colors` assigns a distinct `tab:` colour to each model so that train and test curves for the same model share a colour and are distinguished only by line style.
* `linestyle='-'` for test curves and `linestyle='--'` for train curves — solid lines carry the primary interpretation (generalisation performance), dashed lines show training error as reference.
* Iterating `zip(results.items(), colors)` keeps model name, data dict, and colour in sync without indexing.
* `ax.legend(loc='upper right', fontsize=8, ncol=2)` — two-column legend fits all eight entries without obscuring the curves.
* `plt.tight_layout()` adjusts subplot margins to prevent label clipping.

### Q7.II – Learning Curve Experiment: N = 5, 10, ..., 500
*GitHub issue: #30*

In [ ]:
training_sizes = list(range(5, 505, 5))
n_reps = 10

# Factory functions so each repetition gets a fresh, unfitted model
model_factories = {
    'Logistic Regression': lambda: LogisticRegression(max_iter=10_000),
    'Naive Bayes':         lambda: BayesianClassifier(shared_cov=False, cond_ind=True),
    'BC Shared Cov':       lambda: BayesianClassifier(shared_cov=True,  cond_ind=False),
    'BC Non-Shared Cov':   lambda: BayesianClassifier(shared_cov=False, cond_ind=False),
}

# Only use sizes that fit within the available training data
valid_sizes = [N for N in training_sizes if N < len(X_train)]

# results[name]['train'/'test'] = list of mean error rates, one per valid size
results = {name: {'train': [], 'test': []} for name in model_factories}

for N in valid_sizes:
    rep_train = {name: [] for name in model_factories}
    rep_test  = {name: [] for name in model_factories}
    for _ in range(n_reps):
        # Sample one training set of size N; all models share it (inner loop = repetitions)
        X_tr, _, y_tr, _ = train_test_split(X_train, y_train, train_size=N, shuffle=True)
        for name, factory in model_factories.items():
            m = factory()
            m.fit(X_tr, y_tr)
            # Use zero_one_loss (Activity 3.2/3.3 convention) rather than 1 - np.mean(==)
            rep_train[name].append(zero_one_loss(y_tr,    m.predict(X_tr)))
            rep_test[name].append( zero_one_loss(y_test,  m.predict(X_test)))
    for name in model_factories:
        results[name]['train'].append(np.mean(rep_train[name]))
        results[name]['test'].append(np.mean(rep_test[name]))

### Q7.III – Plot Learning Curves
*GitHub issue: #31*

In [ ]:
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

fig, ax = plt.subplots(figsize=(12, 6))
for (name, data_r), color in zip(results.items(), colors):
    ax.plot(valid_sizes, data_r['test'],  color=color, linestyle='-',
            label=f'{name} (test)')
    ax.plot(valid_sizes, data_r['train'], color=color, linestyle='--',
            label=f'{name} (train)')

ax.set_xlabel('Training set size $N$')
ax.set_ylabel('Error rate')
ax.set_title('Learning curves: Logistic Regression vs Bayes Classifier variants\n(Breast Cancer dataset)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

### Q7.IV – Discussion
#### 1. Effect of increasing training data on each classifier
*GitHub issue: #32*

As $N$ increases, every classifier's test error decreases and its train error increases, following the classic learning curve pattern driven by the bias–variance trade-off.

* **Logistic Regression:** At small $N$ the train error is near zero (high variance — the model memorises the few training points) while the test error is high. As $N$ grows, train error rises and test error falls rapidly, and the two converge toward a low asymptote. LR ultimately achieves the lowest test error because it directly models the decision boundary without making assumptions about $p(\mathbf{x})$.

* **Naive Bayes:** Train error is higher at small $N$ than LR (NB is less flexible; its inductive bias prevents memorisation). However, test error drops quickly and reaches a low value early — NB benefits from having fewer parameters to estimate. At large $N$ the test error levels off above LR because the conditional independence assumption $p(\mathbf{x}|c) = \prod_j p(x_j|c)$ is violated in practice: the 30 breast cancer features are correlated, so the model is misspecified.

* **BC Shared Covariance:** Intermediate behaviour — more parameters than NB (one full $p \times p$ shared covariance), slower initial convergence, but the shared-covariance constraint limits the total number of parameters and provides some regularisation. Its asymptotic error is typically between NB and BC non-shared.

* **BC Non-Shared Covariance:** The most parameter-heavy variant (two full $30 \times 30$ covariance matrices). At small $N$ the covariance estimates are unreliable (singular matrices are likely when $N < p = 30$), producing high test error. At large $N$ it can converge to a good solution if the class-conditional distributions are truly non-Gaussian — but in practice it tends to lag behind LR.

#### 2. Which classifier is best suited for small vs large training sets?
*GitHub issue: #33*

* **Small $N$:** Naive Bayes is generally best suited. It has the fewest parameters among the four models and its strong independence assumption acts as a regulariser, keeping variance low even when little data is available. The bias introduced by the independence assumption is usually tolerable when estimates of more complex structures (full covariances) would be far too noisy.

* **Large $N$:** Logistic Regression is best suited. As $N$ grows, the advantage of NB's inductive bias shrinks — with enough data, all models can estimate their parameters reliably — and LR's discriminative nature takes over. Because LR models $p(y \mid \mathbf{x})$ directly without spending capacity on modelling $p(\mathbf{x})$, it uses its parameters more efficiently and tends to achieve a lower asymptotic error rate.

#### 3. Justify observations via model assumptions and parameter count
*GitHub issue: #34*

The key driver of the observed behaviour is the **number of free parameters** each model must estimate and the **assumptions** it makes to reduce that count.

For the breast cancer dataset ($p = 30$ features, $K = 2$ classes):

| Model | Parameters to estimate | Notes |
|---|---|---|
| Logistic Regression | $p + 1 = 31$ | weight vector + intercept |
| Naive Bayes | $2Kp = 120$ | $K$ mean vectors + $K$ diagonal covariances |
| BC Shared Cov | $Kp + \frac{p(p+1)}{2} \approx 525$ | $K$ means + one full symmetric covariance |
| BC Non-Shared Cov | $Kp + K\frac{p(p+1)}{2} \approx 990$ | $K$ means + $K$ full covariances |

**Why NB dominates at small $N$:** With only $N = 5$ samples and 990 parameters to estimate for BC non-shared, the model is vastly underdetermined — covariance matrices are singular and predictions are unreliable. NB's independence assumption collapses the covariance to a diagonal, reducing the effective parameter count dramatically. The additional bias from assuming independence is small relative to the variance reduction gained.

**Why LR dominates at large $N$:** LR is a discriminative model — it directly learns the boundary $p(y \mid \mathbf{x})$ using only $p + 1$ parameters. It does not model the marginal distribution $p(\mathbf{x})$, which is irrelevant for classification and wastes capacity in generative models. As $N \to \infty$, LR's decision surface (a hyperplane in the original feature space) is flexible enough to correctly classify the breast cancer data, while the generative models are constrained by their distributional assumptions (Gaussianity, shared or independent covariances) that may not hold exactly. This is consistent with the theoretical result of Ng and Jordan (2002), who showed that for Gaussian class conditionals, NB converges to its asymptotic error faster than LR but LR achieves a lower asymptotic error when the independence assumption is violated.

---
## Tests
### `BayesianClassifier` and Learning Curve Results Test Suite

In [ ]:
print("=== BayesianClassifier checks ===")

# Perfectly separable 2-class data in 2D
X_bc = np.array([[0.0, 0.0], [0.1, 0.0], [0.0, 0.1],   # class 0
                 [5.0, 5.0], [5.1, 5.0], [5.0, 5.1]])   # class 1
y_bc = np.array([0, 0, 0, 1, 1, 1])

bc = BayesianClassifier(shared_cov=False, cond_ind=True)
result = bc.fit(X_bc, y_bc)
assert result is bc, "fit() should return self"
print("  PASS  fit() returns self")

assert np.array_equal(bc.classes_, [0, 1]), f"Expected classes_ = [0,1], got {bc.classes_}"
print("  PASS  classes_ is set correctly after fit")

assert np.allclose(bc.class_priors_, [0.5, 0.5]), \
    f"Expected equal priors [0.5, 0.5], got {bc.class_priors_}"
print("  PASS  class_priors_ are 0.5/0.5 for balanced classes")

assert bc.cond_means_.shape == (2, 2), f"Expected cond_means_ shape (2,2), got {bc.cond_means_.shape}"
assert bc.cond_covs_.shape  == (2, 2, 2), f"Expected cond_covs_ shape (2,2,2), got {bc.cond_covs_.shape}"
print("  PASS  cond_means_ and cond_covs_ have correct shapes (k, p) and (k, p, p)")

# cond_ind=True → covariance matrices must be diagonal
for c in range(2):
    off_diag = bc.cond_covs_[c] - np.diag(np.diag(bc.cond_covs_[c]))
    assert np.allclose(off_diag, 0), \
        f"cond_ind=True: class {c} covariance should be diagonal, got {bc.cond_covs_[c]}"
print("  PASS  cond_ind=True produces diagonal covariance matrices")

# shared_cov=True → all classes share the same covariance matrix
bc_shared = BayesianClassifier(shared_cov=True, cond_ind=False).fit(X_bc, y_bc)
assert np.allclose(bc_shared.cond_covs_[0], bc_shared.cond_covs_[1]), \
    "shared_cov=True: all classes should have the same covariance matrix"
print("  PASS  shared_cov=True gives identical covariance matrices for all classes")

# predict_proba output shape and validity
proba = bc.predict_proba(X_bc)
assert proba.shape == (6, 2), f"Expected predict_proba shape (6,2), got {proba.shape}"
print("  PASS  predict_proba output shape is (n_samples, n_classes)")

assert np.allclose(proba.sum(axis=1), 1.0), \
    f"Probabilities should sum to 1 per sample, got {proba.sum(axis=1)}"
print("  PASS  predict_proba rows sum to 1")

assert np.all(proba >= 0) and np.all(proba <= 1), "All probabilities should be in [0, 1]"
print("  PASS  all probabilities are in [0, 1]")

# predict on perfectly separable data should be correct
preds = bc.predict(X_bc)
assert preds.shape == y_bc.shape, f"predict() shape mismatch: {preds.shape}"
assert np.array_equal(preds, y_bc), f"Expected {y_bc}, got {preds}"
print("  PASS  predict() correctly classifies perfectly separable data")

# All three BC variants should produce valid predictions on the breast cancer data
for variant_name, variant in [
    ('Naive Bayes',       BayesianClassifier(shared_cov=False, cond_ind=True)),
    ('BC Shared Cov',     BayesianClassifier(shared_cov=True,  cond_ind=False)),
    ('BC Non-Shared Cov', BayesianClassifier(shared_cov=False, cond_ind=False)),
]:
    variant.fit(X_train, y_train)
    p = variant.predict_proba(X_test)
    assert p.shape == (len(X_test), 2),       f"{variant_name}: predict_proba shape wrong"
    assert np.allclose(p.sum(axis=1), 1.0),   f"{variant_name}: proba rows don't sum to 1"
    assert set(variant.predict(X_test)) <= {0, 1}, f"{variant_name}: predict returned invalid class"
print("  PASS  all three BC variants produce valid outputs on the breast cancer test set")

print("\n=== Learning curve results dict checks ===")

n_valid = len(valid_sizes)
for name in model_factories:
    assert len(results[name]['train']) == n_valid, \
        f"results['{name}']['train'] should have {n_valid} entries, got {len(results[name]['train'])}"
    assert len(results[name]['test'])  == n_valid, \
        f"results['{name}']['test'] should have {n_valid} entries, got {len(results[name]['test'])}"
print(f"  PASS  each model has {n_valid} mean error values (one per training size)")

for name in model_factories:
    assert all(0 <= v <= 1 for v in results[name]['train']), \
        f"Train error rates for '{name}' contain values outside [0,1]"
    assert all(0 <= v <= 1 for v in results[name]['test']), \
        f"Test error rates for '{name}' contain values outside [0,1]"
print("  PASS  all train and test error rates are in [0, 1]")

# At large N all models should achieve reasonable test accuracy (error < 0.3)
for name in model_factories:
    final_test_err = results[name]['test'][-1]
    assert final_test_err < 0.3, \
        f"'{name}' final test error {final_test_err:.3f} is unexpectedly high (> 0.3)"
print("  PASS  all models achieve test error < 0.30 at maximum training size")

# Logistic Regression train error should generally rise with N (less memorisation)
lr_train = results['Logistic Regression']['train']
assert lr_train[-1] > lr_train[0], \
    "LR train error should increase with N (K=1 memorises; large N cannot)"
print("  PASS  Logistic Regression train error increases with N (consistent with VC theory)")

print("\nAll tests passed.")